# **8.3 Enhancing Predictive Models with Survey Data I**  


## Predicting 3rd-Semester Status (Retention) with XGBoost

In earlier notebooks, you built supervised learning models to predict **3rd-semester status** using student academic and demographic data. In this notebook, you’ll extend that workflow by using a dataset that **also includes survey-derived features** (including coded survey variables and text-derived components).

**Why this matters in higher ed:** survey data help you model factors that administrative records can’t fully capture—students’ experiences, perceptions, support needs, and stressors. When used responsibly, these models can support earlier, more targeted interventions.

**Guiding questions (keep these in mind as you work):**
1. *Do survey-related features improve prediction of SEM_3_STATUS beyond academic history alone?*
2. *Which feature groups (academic performance, demographics, survey/text components) appear most influential—and what might that imply for support strategies?*


## Table of Contents
1. Understand the dataset: What’s in ML_Survey_training/testing?
2. Prepare: Load data and define X and y
3. Train: Fit an XGBoost model for SEM_3_STATUS
4. Evaluate: Performance on testing data
5. Interpret: What features mattered most?
6. Interpret by feature group: Academic vs Demographics vs Survey/Text
7. Wrap-up: Turning model insights into institutional action


---
## 1. Understand the Dataset

You will load two files that are assumed to already exist:
- **ML_Survey_training.csv**
- **ML_Survey_testing.csv**

Each file contains a merged dataset that combines:
- **Academic performance** (e.g., HS_GPA, GPA_1, GPA_2, DFW rates, units attempted)
- **Demographics** (one-hot encoded gender, race/ethnicity, first-gen status)
- **Survey/text-derived features** (e.g., coded survey variables and/or text components like TEXT_PC1 … TEXT_PC33)

The target variable is:

- **SEM_3_STATUS**: an indicator of 3rd-semester status (retained vs not retained)

> Note: As in the earlier classification notebooks, we treat SEM_3_STATUS as the supervised learning target.


In [ ]:
# If needed, install xgboost (uncomment if your environment does not already include it)
# !pip -q install xgboost


In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay
)

import matplotlib.pyplot as plt


---
## 2. Prepare: Load Training and Testing Data

We’ll load the two datasets, confirm the target column exists, and split into:

- **X** (features)  
- **y** (target: SEM_3_STATUS)

Because the dataset is already “ML-ready” (numeric columns), we can directly train an XGBoost classifier.


In [ ]:
# Update paths if needed (these are assumed to be available)
train_path = "ML_Survey_training.csv"
test_path  = "ML_Survey_testing.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print("Training shape:", train_df.shape)
print("Testing shape: ", test_df.shape)

train_df.head()


In [ ]:
# Confirm the target column exists
target_col = "SEM_3_STATUS"
assert target_col in train_df.columns, f"Missing target column: {target_col}"
assert target_col in test_df.columns,  f"Missing target column: {target_col}"

# Separate features and target
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test  = test_df.drop(columns=[target_col])
y_test  = test_df[target_col]

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test: ", X_test.shape,  "| y_test: ", y_test.shape)


---
## 3. Train: XGBoost Model

XGBoost is a powerful tree-based ensemble method that often performs well on structured tabular data (like institutional data). It can capture nonlinear relationships and interactions across many features—useful when combining academic, demographic, and survey-based signals.

In this first pass, we’ll start with a reasonable baseline configuration and fit the model.


In [ ]:
from xgboost import XGBClassifier

# A practical baseline configuration (you can tune later)
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)


---
## 4. Evaluate: Model Performance on Testing Data

We’ll evaluate performance using standard classification metrics:
- Accuracy
- Precision, Recall, F1
- Confusion Matrix
- ROC AUC (when the target is binary)

> Reminder: Metrics help you understand prediction quality, but they do **not** automatically tell you whether a model is appropriate for intervention. That requires governance, fairness checks, and thoughtful use.


In [ ]:
# Predictions
y_pred = xgb_model.predict(X_test)

# Predicted probabilities (useful for ROC AUC and risk scoring)
y_proba = xgb_model.predict_proba(X_test)[:, 1] if hasattr(xgb_model, "predict_proba") else None

# Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")
print(f"F1-score:  {f1:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix: XGBoost on SEM_3_STATUS")
plt.show()


In [ ]:
# ROC AUC (only if binary target and probabilities available)
if y_proba is not None and len(np.unique(y_test)) == 2:
    auc = roc_auc_score(y_test, y_proba)
    print(f"ROC AUC: {auc:.3f}")
    RocCurveDisplay.from_predictions(y_test, y_proba)
    plt.title("ROC Curve: XGBoost on SEM_3_STATUS")
    plt.show()
else:
    print("ROC AUC skipped (requires binary target and predict_proba).")


---
## 5. Interpret: What Features Mattered Most?

XGBoost can provide feature importance scores. These do **not** tell you “causal impact,” but they are useful for:
- identifying which variables the model relied on most,
- spotting whether survey/text features are contributing meaningfully,
- and generating hypotheses for institutional action.

We’ll start by extracting the top features and visualizing them.


In [ ]:
# Feature importance from the trained model
importances = xgb_model.feature_importances_
feature_names = X_train.columns

fi = (pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True))

fi.head(20)


In [ ]:
# Plot top 20 features
top_n = 20
fi_top = fi.head(top_n).iloc[::-1]

plt.figure(figsize=(8, 6))
plt.barh(fi_top["feature"], fi_top["importance"])
plt.title(f"Top {top_n} Feature Importances (XGBoost)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


---
## 6. Interpret by Feature Group: Academic vs Demographics vs Survey/Text

A key goal of this notebook is to interpret **how different feature types contributed** to prediction.

We’ll group features into:
1. **Academic performance** (GPA, DFW rates, units attempted, etc.)
2. **Demographics** (one-hot encoded gender, race/ethnicity, first-gen status)
3. **Survey/Text components** (e.g., TEXT_PC1 … TEXT_PC33 and any survey-derived codes)

Then we’ll compare the *total importance* contributed by each group.

> This is a high-level way to answer: *Did survey/text information add signal beyond academics and demographics?*


In [ ]:
# Define feature groups using name-based rules
academic_prefixes = ["HS_GPA", "GPA_", "DFW_RATE_", "UNITS_ATTEMPTED_"]
demo_prefixes = ["GENDER_", "RACE_ETHNICITY_", "FIRST_GEN_STATUS_"]
text_prefixes = ["TEXT_PC"]  # update if you have other survey-derived prefixes (e.g., SURVEY_, Q_, SCALE_)

def assign_group(col):
    if any(col.startswith(p) for p in academic_prefixes):
        return "Academic performance"
    if any(col.startswith(p) for p in demo_prefixes):
        return "Demographics"
    if any(col.startswith(p) for p in text_prefixes):
        return "Survey/Text components"
    # Anything else: keep visible rather than hiding it
    return "Other (check)"

fi["group"] = fi["feature"].apply(assign_group)

group_importance = (fi.groupby("group")["importance"]
                      .sum()
                      .sort_values(ascending=False)
                      .reset_index())

group_importance


In [ ]:
# Plot group importance
plt.figure(figsize=(7, 4))
plt.bar(group_importance["group"], group_importance["importance"])
plt.title("Total Feature Importance by Group (XGBoost)")
plt.ylabel("Total importance (sum)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


### 6.1 Reading the Group Importance Results (Institutional Lens)

Use the chart above to guide interpretation.

- If **Academic performance** dominates: that suggests retention prediction is primarily driven by prior grades/credits/DFW patterns (common in many institutional datasets).
- If **Survey/Text components** contribute meaningfully: that suggests student experience variables are adding signal—potentially useful for designing *earlier* supports that are not purely academic.
- If **Demographics** show high importance: treat this as a prompt for careful review. Demographic features can be predictive due to structural inequities; models should be governed and audited to avoid reinforcing inequity.

**Checkpoint reflection:**  
If survey/text components matter, what might that imply about the kinds of interventions that could help students persist?


---
## 7. Going Deeper: Which Survey/Text Components Matter Most?

If TEXT_PC variables represent text-derived components (for example, from TF-IDF → SVD/PCA), they are “compressed” summaries of language patterns. Individual PCs won’t have intuitive meanings by themselves, but you *can* still see whether they matter.

Below, we list the most important survey/text-related features.


In [ ]:
# Top survey/text components (customize this filter if you have other survey prefixes)
survey_text_top = (fi[fi["group"] == "Survey/Text components"]
                   .sort_values("importance", ascending=False)
                   .head(15)
                   .reset_index(drop=True))

survey_text_top


---
## 8. Wrap-Up: Turning Model Insights into Institutional Action

In this notebook, you:
- loaded a merged survey + academic + demographic dataset,
- trained an XGBoost model to predict **SEM_3_STATUS**,
- evaluated predictive performance,
- and interpreted feature contributions both at the individual feature level and at the feature-group level.

**Key takeaway:** Even when academic variables are strong predictors, survey-derived features can add valuable signal about student experience—and those signals may point to more actionable intervention strategies.

### What’s next
In the next notebook, you will deepen interpretability by:
- tuning the model,
- exploring probability thresholds for “risk” flags,
- and (optionally) introducing model-agnostic explanation tools for clearer insights.

**Final reflection:**  
If you could share one insight from your feature-group analysis with an advising or student success team, what would it be?
